####DAY 7 (26/02/26) – MLflow Tracking & Model Registry
####🏗️ Architecture & Strategy
Welcome to Day 7! Yesterday, we trained models and logged them. Today, we transition into formal MLOps . Training a model in a notebook is easy; tracking its lineage, comparing it against other models, and securely versioning it for a production engineering team is the hard part.

####Our Strategy:

* **The "Fix the Warnings" Run**: Yesterday, MLflow warned us about a missing "Model Signature" and "Input Example." Today, we will train one final competitor model (Gradient Boosted Trees) and perfectly log it with a strict schema signature to solve those warnings.

* **Programmatic Metrics Comparison**: Instead of just looking at the Databricks UI, we will use the MlflowClient Python API to programmatically search our experiment, query all the models we've trained (Logistic Regression, Random Forest, GBT), and extract the ultimate winner based on the highest AUC score.

* **Model Versioning (Unity Catalog)**: Once we programmatically identify the best run, we will register it into the Unity Catalog Model Registry. We will assign it a "Champion" alias, marking it as the official model ready for production deployment.

####Training GBT & Enforcing Model Signatures
Let's train a Gradient Boosted Tree (GBT) model. This time, we will explicitly define a `signature` so MLflow knows exactly what data types to expect in production.    

(Note: We are re-initializing the environment variables so this cell can run independently!)

In [0]:
import os
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from mlflow.models.signature import infer_signature

# 1. Environment Setup 
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
volume_name = "ml_assets"
mlflow_tmp_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/mlflow_staging"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# ⚠️ THE ULTIMATE SECURITY FIX: Force the cluster OS to use the UC Volume globally
os.environ["MLFLOW_DFS_TMP"] = mlflow_tmp_path

# We load the raw data, but DO NOT transform it yet!
train_df = spark.table("gold_train_set")
test_df = spark.table("gold_test_set")
feature_cols = ["total_events", "view_count", "cart_count"]

# Setup Evaluator & MLflow Experiment
evaluator = BinaryClassificationEvaluator(labelCol="purchased", metricName="areaUnderROC")
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name = f"/Users/{username}/Day6_Purchase_Prediction"
mlflow.set_experiment(experiment_name)

print("🚀 Training Competitor: Gradient Boosted Trees (GBT Pipeline)...")

with mlflow.start_run(run_name="GBT_Pipeline_Production"):
    mlflow.log_param("model_type", "GBT_Pipeline")
    
    # 2. Define the Pipeline Stages
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
    gbt = GBTClassifier(featuresCol="features", labelCol="purchased", maxIter=20, seed=42)
    
    # Chain them together into a single Pipeline
    pipeline = Pipeline(stages=[assembler, gbt])
    
    # 3. Fit the entire pipeline on the RAW dataframe
    pipeline_model = pipeline.fit(train_df)
    
    # 4. Evaluate Model
    predictions = pipeline_model.transform(test_df)
    auc_score = evaluator.evaluate(predictions)
    mlflow.log_metric("test_auc", auc_score)
    
    # 5. Generate Model Signature & Input Example using the RAW data
    input_example = train_df.select(feature_cols).limit(1).toPandas()
    output_example = predictions.select("prediction").limit(1).toPandas()
    signature = infer_signature(input_example, output_example)
    
    # 6. Log the Pipeline Model with strict MLOps standards
    mlflow.spark.log_model(
        spark_model=pipeline_model, 
        artifact_path="gbt_pipeline_model", 
        dfs_tmpdir=mlflow_tmp_path,
        signature=signature,           
        input_example=input_example    
    )    
    
    print(f"   🏆 GBT AUC: {auc_score:.4f}")
    print("   ✅ Pipeline Model logged securely WITH signature and input example.")

####Programmatic Metrics Comparison
Now that we have multiple models logged to our experiment (from Day 6 and Day 7), let's use the MLflow API to query the tracking server and find the highest-performing model automatically.

In [0]:
# ---------------------------------------------------------
# PROGRAMMATIC EXPERIMENT COMPARISON
# ---------------------------------------------------------
print("📊 Searching MLflow Experiment for the best model...")

# 1. Search the MLflow tracking server for all runs in our experiment
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.test_auc DESC"] # Sort by AUC, highest first
)

# 2. Clean up the dataframe for display
# We extract just the important columns so business stakeholders can read it
display_df = runs_df[[
    "run_id", 
    "tags.mlflow.runName", 
    "params.model_type", 
    "metrics.test_auc", 
    "status"
]].head(5) # Top 5 runs

print("🏆 Top Model Runs Ranked by AUC:")
display(display_df)

# 3. Dynamically extract the winning run ID for the next step
best_run_id = runs_df.iloc[0]["run_id"]
best_run_name = runs_df.iloc[0]["tags.mlflow.runName"]
best_run_auc = runs_df.iloc[0]["metrics.test_auc"]

print(f"\n🥇 THE WINNER IS: '{best_run_name}' (Run ID: {best_run_id}) with an AUC of {best_run_auc:.4f}")

Model Registry & Versioning (Unity Catalog)
We have identified the best model. Now, we register it to Unity Catalog. This promotes the model from a "data science experiment" to an official "engineering asset" that can be governed, audited, and deployed.

In [0]:
from mlflow.tracking import MlflowClient

# ---------------------------------------------------------
# UNITY CATALOG MODEL REGISTRY (Dynamic)
# ---------------------------------------------------------
print("⚙️ Dynamically resolving the winning model's artifact path...")

# 1. Fetch the model_type we logged for the winning run
winning_model_type = runs_df.iloc[0]["params.model_type"]

# 2. Map the model type to the exact artifact path we used when logging
if winning_model_type == "GBT_Pipeline":
    artifact_path = "gbt_pipeline_model"
elif winning_model_type == "RandomForest":
    artifact_path = "rf_best_model"
else:
    artifact_path = "lr_model"

print(f"   ➤ Winning Model Type: {winning_model_type}")
print(f"   ➤ Resolved Artifact Path: {artifact_path}")

# 3. Construct the exact URI
uc_model_name = f"{catalog_name}.{schema_name}.purchase_prediction_classifier"
model_uri = f"runs:/{best_run_id}/{artifact_path}" 

print(f"\n📦 Registering winning model to Unity Catalog: {uc_model_name}...")

# 4. Register the Model
model_version = mlflow.register_model(
    model_uri=model_uri, 
    name=uc_model_name
)

print(f"   ✅ Registered as Version {model_version.version}")

# 5. Apply Alias using MlflowClient
client = MlflowClient()
client.set_registered_model_alias(
    name=uc_model_name, 
    alias="champion", 
    version=model_version.version
)

print(f"   👑 Assigned 'champion' alias to Version {model_version.version}.")
print("\n🎉 Day 7 Complete! Your model is fully production-ready and version-controlled.")

# 💡 UI VERIFICATION INSTRUCTIONS:
# 1. Go to "Catalog" on the left sidebar.
# 2. Navigate to course_catalog -> ecommerce_governed.
# 3. Look for "purchase_prediction_classifier" with a little 🧠 icon.
# 4. Click it to see your formally governed model asset!

####Key Learnings & Interview Talking Points
If an interviewer asks you about Model Lifecycle Management or MLOps, use these points:

* **Model Signatures & Contracts**: "I strictly enforce Model Signatures using infer_signature(). By logging models with a defined input schema and example, I prevent downstream deployment crashes caused by data type mismatches when software engineers integrate my models."

* **Programmatic Metric Comparison**: "Instead of manually reviewing the MLflow UI, I automate model selection in my pipelines using mlflow.search_runs(). This allows CI/CD systems to dynamically identify and promote the highest-scoring model (e.g., via metrics.test_auc DESC) without human intervention."

* **Unity Catalog Model Registry**: "I register my finalized models into Unity Catalog. This unifies my data governance and model governance under the same security model, providing full auditability of which dataset generated which model."

* **Aliasing over Versioning**: "When transitioning models to production, I utilize Model Aliases (like champion or challenger) rather than hardcoding version numbers. This decouples the Data Science iteration cycle from the software engineering API, allowing me to hot-swap models without breaking production code